# SentinelML Autoencoder Anomaly Detection

Unsupervised fraud anomaly detection trained only on normal transactions from the training split.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.autoencoder import (
    compare_autoencoder_vs_xgboost,
    compute_reconstruction_error,
    train_autoencoder,
    tune_threshold,
)
from src.train_models import load_processed_splits, train_xgboost

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = load_processed_splits(
    PROJECT_ROOT / "data" / "processed"
)

In [ ]:
autoencoder_model, scaler = train_autoencoder(X_train, y_train)

## Training Loss

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(autoencoder_model.loss_history_) + 1), autoencoder_model.loss_history_)
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Autoencoder Training Loss")
ax.grid(alpha=0.25)
plt.show()

## Reconstruction Error Distribution

In [ ]:
val_errors = compute_reconstruction_error(autoencoder_model, scaler, X_val)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(val_errors[y_val == 0], bins=80, alpha=0.6, density=True, label="Normal")
ax.hist(val_errors[y_val == 1], bins=80, alpha=0.6, density=True, label="Fraud")
ax.set_xlabel("Reconstruction Error")
ax.set_ylabel("Density")
ax.set_title("Validation Reconstruction Error by Class")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Threshold Tuning

In [ ]:
threshold, threshold_metrics = tune_threshold(autoencoder_model, scaler, X_val, y_val)
threshold_metrics["threshold_comparison"]

## Autoencoder vs XGBoost Catch Overlap

In [ ]:
xgboost_model, xgboost_metrics = train_xgboost(X_train, y_train, X_val, y_val)
autoencoder_results = {
    "model": autoencoder_model,
    "scaler": scaler,
    "threshold": threshold,
    "metrics": threshold_metrics,
}
overlap = compare_autoencoder_vs_xgboost(autoencoder_results, xgboost_model, X_val, y_val)
pd.DataFrame([overlap])